# 08 — Three-Body ICNN Fun Case

This notebook adds a visually interesting stress-test case to the main ICNN Lyapunov project.

Important framing:

- This is **not** the main proof experiment.
- It is a fun nonlinear 12D rollout test.
- The system is a damped + softened 2D three-body problem, so it is more compatible with a stable Lyapunov-to-origin model than the conservative chaotic three-body problem.
- We compare:
  - baseline MLP dynamics,
  - ICNN Lyapunov stable dynamics,
  - true numerical rollout.

State:

```text
[x1, y1, x2, y2, x3, y3, vx1, vy1, vx2, vy2, vx3, vy3]
```

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for p in [start, *start.parents]:
        if (p / "stable_icnn_physics").exists():
            return p
    raise RuntimeError("Start Jupyter from the DeepLearningFTN repository.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from stable_icnn_physics import BaselineDynamicsMLP, build_stable_model
from stable_icnn_physics.data import (
    dataset_base_name,
    generate_derivative_data,
    load_dataset,
    save_dataset,
    tensor_dataset,
)
from stable_icnn_physics.eval import (
    autoregressive_rollout_model,
    lyapunov_decrease_values,
    rollout_error,
    rollout_system,
)
from stable_icnn_physics.train import evaluate_derivative_mse, train_derivative_model
from stable_icnn_physics.three_body import DampedSoftenedThreeBody2D
from stable_icnn_physics.three_body_viz import (
    animate_three_body_comparison,
    plot_three_body_paths,
    save_gif,
)

torch.set_float32_matmul_precision("high")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CACHE_DIR = REPO_ROOT / "data" / "cache"
CKPT_DIR = REPO_ROOT / "checkpoints"
RESULTS_DIR = REPO_ROOT / "results"
PLOTS_DIR = RESULTS_DIR / "three_body_plots"

for d in [CACHE_DIR, CKPT_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_ROOT)
print("device:", DEVICE, "| torch:", torch.__version__)

## Configuration

In [ ]:
SEED = 0
TOLERANCE = 1e-5

system = DampedSoftenedThreeBody2D(
    gravity=1.0,
    damping=0.08,
    softening=0.08,
    position_scale=1.0,
    velocity_scale=0.9,
    perturbation=0.12,
)

DT = 0.01
STEPS = 500
TRAIN_SAMPLES = 80_000
TEST_SAMPLES = 10_000
EVAL_TRAJS = 8

HIDDEN = 256
DEPTH = 3
LYAPUNOV_HIDDEN = 128
ALPHA = 1e-5
EPOCHS = 250
BATCH_SIZE = 512
LR = 1e-3

TAG = "three_body_damped_softened_icnn"

print(system.metadata())

## Generate/load derivative dataset

In [ ]:
train_path = CACHE_DIR / dataset_base_name(
    system, split="train", n_samples=TRAIN_SAMPLES, seed=SEED, dataset_type="derivative"
)
test_path = CACHE_DIR / dataset_base_name(
    system, split="test", n_samples=TEST_SAMPLES, seed=SEED, dataset_type="derivative"
)

if not train_path.exists():
    print("generating train:", train_path.name)
    x_train, y_train = generate_derivative_data(system, TRAIN_SAMPLES, split="train", seed=SEED)
    save_dataset(train_path, x_train, y_train, metadata={"system": system.metadata(), "split": "train"})
else:
    print("reusing train:", train_path.name)

if not test_path.exists():
    print("generating test:", test_path.name)
    x_test, y_test = generate_derivative_data(system, TEST_SAMPLES, split="test", seed=SEED)
    save_dataset(test_path, x_test, y_test, metadata={"system": system.metadata(), "split": "test"})
else:
    print("reusing test:", test_path.name)

x_train, y_train = load_dataset(train_path)
x_test, y_test = load_dataset(test_path)

train_ds = tensor_dataset(x_train, y_train)
test_ds = tensor_dataset(x_test, y_test)

print("train:", x_train.shape, y_train.shape)
print("test:", x_test.shape, y_test.shape)

## Build/train models

In [ ]:
def make_stable():
    return build_stable_model(
        dim=system.state_dim,
        hidden=HIDDEN,
        depth=DEPTH,
        lyapunov_hidden=LYAPUNOV_HIDDEN,
        lyapunov_eps=0.01,
        alpha=ALPHA,
        rehu_width=0.01,
    )

def make_baseline():
    return BaselineDynamicsMLP(dim=system.state_dim, hidden=HIDDEN, depth=DEPTH)

stable_ckpt = CKPT_DIR / f"{TAG}_stable.pt"
baseline_ckpt = CKPT_DIR / f"{TAG}_baseline.pt"

stable = make_stable()
baseline = make_baseline()

if stable_ckpt.exists():
    print("loading stable:", stable_ckpt)
    stable.load_state_dict(torch.load(stable_ckpt, map_location=DEVICE, weights_only=True)["model_state"])
else:
    print("training stable ICNN model...")
    train_derivative_model(
        stable,
        train_ds,
        test_ds,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        device=DEVICE,
        checkpoint_path=stable_ckpt,
        print_every=max(1, EPOCHS // 10),
        use_amp=False,
    )

if baseline_ckpt.exists():
    print("loading baseline:", baseline_ckpt)
    baseline.load_state_dict(torch.load(baseline_ckpt, map_location=DEVICE, weights_only=True)["model_state"])
else:
    print("training baseline model...")
    train_derivative_model(
        baseline,
        train_ds,
        test_ds,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        device=DEVICE,
        checkpoint_path=baseline_ckpt,
        print_every=max(1, EPOCHS // 10),
        use_amp=(DEVICE == "cuda"),
    )

stable.to(DEVICE).eval()
baseline.to(DEVICE).eval()

## Rollout evaluation

In [ ]:
x0 = system.sample_initial_conditions(EVAL_TRAJS, split="test", seed=SEED + 123)
true_traj = rollout_system(system, x0, steps=STEPS, dt=DT)

stable_traj = autoregressive_rollout_model(
    stable, x0, steps=STEPS, dt=DT, device=DEVICE, wrap_fn=system.wrap_state
)
baseline_traj = autoregressive_rollout_model(
    baseline, x0, steps=STEPS, dt=DT, device=DEVICE, wrap_fn=system.wrap_state
)

err_stable = rollout_error(system, true_traj, stable_traj).mean(axis=1)
err_baseline = rollout_error(system, true_traj, baseline_traj).mean(axis=1)

dmse_stable = evaluate_derivative_mse(stable, test_ds, device=DEVICE)
dmse_baseline = evaluate_derivative_mse(baseline, test_ds, device=DEVICE)

decrease = lyapunov_decrease_values(stable, x_test[:2048], device=DEVICE).ravel()

summary = {
    "experiment": TAG,
    "system": system.metadata(),
    "dt": DT,
    "steps": STEPS,
    "eval_trajs": EVAL_TRAJS,
    "derivative_mse_stable": float(dmse_stable),
    "derivative_mse_baseline": float(dmse_baseline),
    "final_rollout_error_stable": float(err_stable[-1]),
    "final_rollout_error_baseline": float(err_baseline[-1]),
    "mean_rollout_error_stable": float(err_stable.mean()),
    "mean_rollout_error_baseline": float(err_baseline.mean()),
    "max_rollout_error_stable": float(err_stable.max()),
    "max_rollout_error_baseline": float(err_baseline.max()),
    "lyapunov_max_violation_projected": float(decrease.max()),
    "lyapunov_fraction_satisfied_projected": float(np.mean(decrease <= TOLERANCE)),
}

summary_path = RESULTS_DIR / f"{TAG}_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("saved:", summary_path)

## Rollout error plots

In [ ]:
t = np.arange(STEPS + 1) * DT

plt.figure(figsize=(8, 4))
plt.plot(t, err_baseline, label="baseline")
plt.plot(t, err_stable, label="ICNN stable")
plt.xlabel("time [s]")
plt.ylabel("mean state error")
plt.title("Three-body rollout error")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOTS_DIR / f"{TAG}_rollout_error.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

plt.figure(figsize=(8, 4))
plt.semilogy(t, err_baseline + 1e-12, label="baseline")
plt.semilogy(t, err_stable + 1e-12, label="ICNN stable")
plt.xlabel("time [s]")
plt.ylabel("mean state error, log scale")
plt.title("Three-body rollout error, log scale")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = PLOTS_DIR / f"{TAG}_rollout_error_log.png"
plt.savefig(path, dpi=160)
plt.show()
print("saved:", path)

## Path visualization

In [ ]:
traj_id = 0

fig, ax = plot_three_body_paths(
    true_traj,
    baseline_traj,
    traj_id=traj_id,
    title="Three-body paths: true vs baseline",
)
path = PLOTS_DIR / f"{TAG}_paths_baseline.png"
fig.savefig(path, dpi=160)
plt.show()
print("saved:", path)

fig, ax = plot_three_body_paths(
    true_traj,
    stable_traj,
    traj_id=traj_id,
    title="Three-body paths: true vs ICNN stable",
)
path = PLOTS_DIR / f"{TAG}_paths_stable.png"
fig.savefig(path, dpi=160)
plt.show()
print("saved:", path)

## Animation

In [ ]:
anim_baseline = animate_three_body_comparison(
    true_traj,
    baseline_traj,
    traj_id=traj_id,
    interval_ms=30,
    title="Three-body true vs baseline",
    trail=120,
)
baseline_gif = RESULTS_DIR / f"{TAG}_true_vs_baseline.gif"
save_gif(anim_baseline, baseline_gif, fps=30)
print("saved:", baseline_gif)

anim_stable = animate_three_body_comparison(
    true_traj,
    stable_traj,
    traj_id=traj_id,
    interval_ms=30,
    title="Three-body true vs ICNN stable",
    trail=120,
)
stable_gif = RESULTS_DIR / f"{TAG}_true_vs_icnn_stable.gif"
save_gif(anim_stable, stable_gif, fps=30)
print("saved:", stable_gif)

## Interpretation

This case is primarily a **visual nonlinear stress test**.

Expected behaviour:

- baseline may fit short-term local dynamics well,
- long rollout can drift quickly because the three-body interaction is sensitive,
- ICNN stable model may damp/regularize the rollout more strongly,
- if ICNN error is higher but boundedness is better, this is still useful for the discussion section.

Do not present this as the main proof of the project. Present it as a fun additional case after the main `mass_spring_damper` and `damped_pendulum_4` experiments.